# For colab environment

In [1]:
# !pip install nltk transformers==4.35.0 torch==2.6.0 torchvision==0.21.0 datasets accelerate==0.24.0 huggingface==0.0.1 datasets==2.14.7

In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [3]:
# !git clone https://github.com/BernardMoy/NLP-PCL-Classification.git

In [4]:
# %cd NLP-PCL-Classification/

In [2]:
!nvidia-smi

Sun Mar  1 22:13:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 32%   45C    P8             20W /  250W |     322MiB /  12288MiB |     32%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load train and validation data set

In [3]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1

# train val split 
train_labels = pd.read_csv('data/train_semeval_parids-labels.csv')["par_id"]
val_labels = pd.read_csv('data/dev_semeval_parids-labels.csv')["par_id"]
df_train = df[df["par_id"].isin(train_labels)].reset_index() 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 


# Text Cleaning

In [4]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_train["text"] = df_train["text"].str.replace(char, "")
    df_val["text"] = df_val["text"].str.replace(char, "")


print(df_train["text"].iloc[55])


# Replace numbers with 0
df_train["text"] = df_train["text"].str.replace(r"\d+", "0", regex=True)
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)

print(df_train["text"].iloc[3])

People who are homeless , those who were once homeless , those working with the homeless and concerned New Zealanders are being asked to share their experiences and solutions to this growing issue with the Cross-Party Homelessness Inquiry . More
Council customers only signs would be displayed . Two of the spaces would be reserved for disabled persons and there would be five P0 spaces and eight P0 ones .


# Oversample the minority class
For each keyword category, inflate the number of positive examples to a certain percentage

In [5]:
POSITIVE_PERCENTAGE = 25


# Find all the unique keywords in the training dataset
keywords = pd.unique(df_train["keyword"])


# Extract the sub-dataset for each keyword
for keyword in keywords:
    subdata = df_train[df_train["keyword"] == keyword]
    rows = subdata.shape[0]


    # Find the number of positive entires x
    subdata_positive = subdata[subdata["bool_labels"] == True]
    positive_rows = subdata_positive.shape[0]


    # Calculate the number of additional samples needed to make the positive class reach the desired percentage
    # (p+x)/(r+x) = POS PERCENTAGE
    n_samples = round((100*positive_rows-POSITIVE_PERCENTAGE*rows)/(POSITIVE_PERCENTAGE-100)*1.0)


    # Sample with replacement from the sub dataset and add new rows
    sampled = subdata_positive.sample(n_samples, replace=True).reset_index(drop=True)
   
    # concat with the main df
    df_train = pd.concat([df_train, sampled], ignore_index=True)


df_train["bool_labels"].value_counts()


bool_labels
False    7581
True     2527
Name: count, dtype: int64

# Add contextual information to the text tokens

In [6]:
def add_info(df): 
    # Append the keyword and country code to the text, and separate them with additional separator tokens
    # Remove dashes in the keyword to match the format in the texts 
    return df["keyword"].str.replace('-', " ") + "<SEP>" + df["country_code"] + "<SEP>" + df["text_cr"]

# Tokenization

In [7]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer = RobertaTokenizer.from_pretrained("roberta-base") 

# define special tokens for separating text 
special_tokens = {"additional_special_tokens": ["<SEP>"]}
tokenizer.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(df): 
    text_with_context = df["text"]

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Convert to pyTorch dataset

In [8]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset(df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["label"].values 
    })

In [9]:
train_dataset = to_dataset(df_train)
val_dataset = to_dataset(df_val) 

# set to torch format 
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Training 

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics 
    accuracy = accuracy_score(labels, predictions) 
    precision = precision_score(labels, predictions, average="macro") 
    recall = recall_score(labels, predictions, average="macro")  
    f1 = f1_score(labels, predictions, average="macro") 

    return {
        "accuracy": accuracy, 
        "precision": precision, 
        "recall": recall, 
        "f1": f1 
    }


In [11]:
# Load roberta sequence classification model 
config = AutoConfig.from_pretrained("roberta-base", num_labels=5)  # Predict the PCL level from 0-4
model = RobertaForSequenceClassification.from_pretrained("roberta-base", config = config)
model.resize_token_embeddings(len(tokenizer)) 

# Core hyperparameters 
BATCH_SIZE = 32
N_EPOCHS = 5 

# Set up training arguments 
training_args = TrainingArguments(
    fp16=True, 
    num_train_epochs=N_EPOCHS, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    warmup_steps=500, 
    save_strategy="epoch",  # low disk space 
    load_best_model_at_end=True, 
    metric_for_best_model='f1',
    logging_steps=50,
    output_dir="./checkpoints/roberta_ensemble", 
    evaluation_strategy="epoch", 
    per_device_eval_batch_size=BATCH_SIZE, 
    per_device_train_batch_size=BATCH_SIZE, 
)

# Set up trainer 
trainer = Trainer(
    model = model, 
    args = training_args, 
    train_dataset=train_dataset, 
    eval_dataset=val_dataset, 
    compute_metrics=compute_metrics
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.bias', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [12]:
trainer.train() 

  0%|          | 0/1580 [00:00<?, ?it/s]

{'loss': 1.561, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.16}
{'loss': 1.2027, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.32}
{'loss': 1.0158, 'learning_rate': 5.9600000000000005e-06, 'epoch': 0.47}
{'loss': 0.8727, 'learning_rate': 7.960000000000002e-06, 'epoch': 0.63}
{'loss': 0.8499, 'learning_rate': 9.960000000000001e-06, 'epoch': 0.79}
{'loss': 0.7959, 'learning_rate': 1.1920000000000001e-05, 'epoch': 0.95}


  0%|          | 0/66 [00:00<?, ?it/s]

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'eval_loss': 0.6272671818733215, 'eval_accuracy': 0.7725752508361204, 'eval_precision': 0.3167697959709194, 'eval_recall': 0.3421430929814079, 'eval_f1': 0.28097053671574485, 'eval_runtime': 17.2615, 'eval_samples_per_second': 121.253, 'eval_steps_per_second': 3.824, 'epoch': 1.0}
{'loss': 0.7286, 'learning_rate': 1.392e-05, 'epoch': 1.11}
{'loss': 0.664, 'learning_rate': 1.5920000000000003e-05, 'epoch': 1.27}
{'loss': 0.6855, 'learning_rate': 1.792e-05, 'epoch': 1.42}
{'loss': 0.672, 'learning_rate': 1.9920000000000002e-05, 'epoch': 1.58}
{'loss': 0.6088, 'learning_rate': 1.9111111111111113e-05, 'epoch': 1.74}
{'loss': 0.5811, 'learning_rate': 1.8185185185185186e-05, 'epoch': 1.9}


...

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.701582133769989, 'eval_accuracy': 0.782608695652174, 'eval_precision': 0.4002120121419791, 'eval_recall': 0.4214060375339771, 'eval_f1': 0.40399108253941557, 'eval_runtime': 17.1781, 'eval_samples_per_second': 121.841, 'eval_steps_per_second': 3.842, 'epoch': 4.0}
{'loss': 0.1829, 'learning_rate': 5.2222222222222226e-06, 'epoch': 4.11}
{'loss': 0.1613, 'learning_rate': 4.296296296296296e-06, 'epoch': 4.27}
{'loss': 0.1413, 'learning_rate': 3.3703703703703705e-06, 'epoch': 4.43}
{'loss': 0.1569, 'learning_rate': 2.4444444444444447e-06, 'epoch': 4.59}
{'loss': 0.1654, 'learning_rate': 1.5185185185185186e-06, 'epoch': 4.75}
{'loss': 0.1466, 'learning_rate': 5.925925925925927e-07, 'epoch': 4.91}


  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.72955322265625, 'eval_accuracy': 0.800764452938366, 'eval_precision': 0.39784568553849065, 'eval_recall': 0.3596483149412094, 'eval_f1': 0.37186189878630166, 'eval_runtime': 17.3106, 'eval_samples_per_second': 120.909, 'eval_steps_per_second': 3.813, 'epoch': 5.0}
{'train_runtime': 1410.6789, 'train_samples_per_second': 35.827, 'train_steps_per_second': 1.12, 'train_loss': 0.4949233085294313, 'epoch': 5.0}


TrainOutput(global_step=1580, training_loss=0.4949233085294313, metrics={'train_runtime': 1410.6789, 'train_samples_per_second': 35.827, 'train_steps_per_second': 1.12, 'train_loss': 0.4949233085294313, 'epoch': 5.0})

In [13]:
trainer.evaluate()

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.701582133769989,
 'eval_accuracy': 0.782608695652174,
 'eval_precision': 0.4002120121419791,
 'eval_recall': 0.4214060375339771,
 'eval_f1': 0.40399108253941557,
 'eval_runtime': 16.5529,
 'eval_samples_per_second': 126.443,
 'eval_steps_per_second': 3.987,
 'epoch': 5.0}

# Save model

In [14]:
trainer.save_model('models/model_roberta_ensemble')